# CV Masterclass 10: Production Augmentations

Welcome to Notebook 10. The pristine academic datasets (ImageNet, COCO) are a lie. They are perfectly lit, high-resolution `.png` files captured by professional cameras.

Production data comes from $10 webcam sensors covered in dust, mounted on vibrating factory machinery, streaming aggressively compressed MJPEG video over weak Wi-Fi.

If you do not simulate this violence during training, your $99\%$ accurate CNN will instantly drop to $30\%$ accuracy in production.

---

## 🎓 Socratic Deep Check
Ponder these questions before we begin:

> *"Mathematically, what destroys a CNN's feature maps when a clear PNG is converted to an aggressively compressed MJPEG video stream?"*

> *"CutMix pastes a region from Image B onto Image A with a label proportional to the pasted area. A student proposes using CutMix at TEST TIME: cut a region from a target image and paste it onto 10 different training images, then average the predictions of the augmented training images (a transductive augmentation). Why does this fail for object detection (where you need precise bounding boxes) but can theoretically work for image classification? What are the two practical failure modes that make test-time CutMix inferior to standard 10-crop TTA in practice?"*

---

## Table of Contents
1. **The MJPEG Assassin:** Discrete Cosine Transforms & Artifacts.
2. **Albumentations:** Building a brutal physical pipeline.
3. **AutoAugment and RandAugment:** Algorithmic simplification of policies.
4. **AugMix:** Mixing Augmentation Chains and JS-Consistency.
5. **Test-Time Augmentation (TTA):** The free accuracy boost.
6. **Class Imbalance Handling:** Explicit class distributions and Focal Loss.
7. **MixUp & CutMix Regularization:** Destroying the spatial prior.


## 1. The MJPEG Assassin

Deep Neural Networks are essentially mathematical edge detectors. 
JPEG compression does not uniformly lower the resolution of an image. It slices the image into $8\times8$ blocks, applies a **Discrete Cosine Transform (DCT)**, and deletes the high-frequency color variations to save bytes.

When stitched back together, this creates microscopic "Block Artifacts"—sharp, artificial grid lines running across the entire image every 8 pixels. 

A human eye ignores these tiny grid lines. But a CNN's first layer is literally a sequence of edge detectors. The CNN latches onto these intense artificial vertical and horizontal lines, completely drowning out the actual physical edges of the objects you want to detect. The feature maps shatter.


In [ ]:
import albumentations as A
import cv2
from matplotlib import pyplot as plt
import numpy as np

# -----------------------------------------------------
# IMPLEMENTATION: The Brutal Edge Pipeline
# -----------------------------------------------------

production_transform = A.Compose([
    # 1. Simulate camera vibration
    A.MotionBlur(blur_limit=15, p=0.3),
    # 2. Simulate ISO sensor noise
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    # 3. Simulate sunlight glare
    A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.5),
    # 4. SIMULATE THE MJPEG ASSASSIN
    A.ImageCompression(quality_lower=30, quality_upper=80, p=0.4),
    # 5. Simulate camera misalignment
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5)
])

# TEST IT
sample_img = np.zeros((256, 256, 3), dtype=np.uint8)
cv2.rectangle(sample_img, (50, 50), (200, 200), (0, 255, 0), -1)
augmented = production_transform(image=sample_img)['image']
print(f"Albumentations applied. Output shape: {augmented.shape}")


### COMMON PITFALLS
*   **Assuming resize interpolations are uniform:** Differing interpolation methods (`cv2.INTER_LINEAR` vs `cv2.INTER_NEAREST`) heavily alter edge sharpness. Using a mismatch between training interpolation and production resizing shatters accuracy.


## 3. AutoAugment and RandAugment

Manually designing augmentation policies (as in the Albumentations pipeline) requires domain expertise and extensive hyperparameter tuning. **AutoAugment** uses reinforcement learning to search for the optimal augmentation policy—but this requires $\sim 15,000$ GPU hours to search on ImageNet.

**RandAugment** is the engineering simplification. It discards the reinforcement learning entirely. Instead, it randomly samples $N$ operations from a fixed set of 14 (AutoContrast, Equalize, Rotate, Solarize, Color, Posterize, Contrast, Brightness, Sharpness, ShearX, ShearY, TranslateX, TranslateY, Cutout) and applies them with a uniform magnitude $M$.

**TWO hyperparameters total:** 
- $N$ (how many ops)
- $M$ (how strong). 
*Default:* $N=2, M=9$. Zero GPU hours needed to search.

RandAugment consistently outperforms manually-designed pipelines by $\sim 0.5\%$ on ImageNet-C (corrupted ImageNet), which strictly measures robustness to distribution shift.


In [ ]:
import torch
from torchvision.transforms import v2
import matplotlib.pyplot as plt

# -----------------------------------------------------
# IMPLEMENTATION: RandAugment
# -----------------------------------------------------
# Using the updated torchvision v2 API
rand_augment = v2.RandAugment(num_ops=2, magnitude=9)

# TEST IT
# Generate a dummy RGB image (CHW)
dummy_tensor = torch.randint(0, 256, (3, 224, 224), dtype=torch.uint8)

# Show diversity: generate 16 augmented versions
fig, axs = plt.subplots(4, 4, figsize=(8, 8))
for i in range(16):
    aug_img = rand_augment(dummy_tensor).permute(1, 2, 0).numpy()
    ax = axs[i//4, i%4]
    ax.imshow(aug_img)
    ax.axis('off')
plt.suptitle("RandAugment (N=2, M=9) Output Grid Diversity", y=1.02)
plt.tight_layout()
plt.show()

print("RandAugment generates highly diverse, randomized corruption samples instantly.")


### COMMON PITFALLS
*   **Applying magnitude $M$ blindly to everything:** Rotations and Translations scale with $M$ fine, but operations like `Solarize` easily destroy semantic information if $M$ is too high. Ensure the magnitude mapping functions are bounded safely.


## 4. AugMix: Mixing Augmentation Chains

Standard augmentations (like Albumentations or RandAugment) are applied **sequentially**—one operation corrupts the image, followed by another. The model only sees one highly corrupted image per training step.

**AugMix** applies $K=3$ SEPARATE chains of random augmentations to the same image. Then, it mixes them together with Dirichlet-sampled weights. The final generated image is a convex combination of the $K$ augmented versions PLUS the original clean image. This yields structurally preserved images that have highly diverse textures/colors.

**Jensen-Shannon Consistency Loss:**
During training with AugMix, the model computes predictions on the Clean image, and the Mixed images. The Jensen-Shannon Consistency Loss enforces that the neural network must output similar mathematical distributions for ALL variations. 
This explicitly penalizes the network heavily if its feature maps are sensitive to augmentation—a vastly stronger robustness signal than standard CE loss.

$$\text{Loss} = \text{CrossEntropy}(p_{clean}, y) + \lambda \cdot \text{JS}(p_{clean}, p_{mix1}, p_{mix2})$$


In [ ]:
import torch.nn.functional as F

# -----------------------------------------------------
# IMPLEMENTATION: JS Consistency Loss for AugMix
# -----------------------------------------------------

def jensen_shannon_divergence(p, q, r):
    # p, q, r are logits. We convert to log_softmax and softmax probabilities
    p_probs = F.softmax(p, dim=1)
    q_probs = F.softmax(q, dim=1)
    r_probs = F.softmax(r, dim=1)
    
    # Calculate the mixture probability
    m = torch.clamp((p_probs + q_probs + r_probs) / 3.0, 1e-7, 1.0)
    
    # Compute KL divergences between each distribution and the mixture
    kl_p = F.kl_div(F.log_softmax(p, dim=1), m, reduction='batchmean')
    kl_q = F.kl_div(F.log_softmax(q, dim=1), m, reduction='batchmean')
    kl_r = F.kl_div(F.log_softmax(r, dim=1), m, reduction='batchmean')
    
    return (kl_p + kl_q + kl_r) / 3.0

# TEST IT
logits_clean = torch.randn(4, 10) # 4 samples, 10 classes
logits_aug1 = logits_clean + torch.randn(4, 10) * 0.5 
logits_aug2 = logits_clean + torch.randn(4, 10) * 0.8

js_loss = jensen_shannon_divergence(logits_clean, logits_aug1, logits_aug2)
# Standard CE loss would be added here
targets = torch.randint(0, 10, (4,))
ce_loss = F.cross_entropy(logits_clean, targets)

total_loss = ce_loss + 12.0 * js_loss # Lambda is typically high (e.g. 12)
print(f"CE: {ce_loss.item():.4f} | JS: {js_loss.item():.4f} | Total: {total_loss.item():.4f}")


### COMMON PITFALLS
*   **Heavy Compute Pipeline:** Generating multiple chains of augmentation dynamically heavily strains the dataloader's CPU workers. Without sufficient CPU multiprocessing, GPU utilization collapses waiting for Augmix batches.


## 5. Test-Time Augmentation (TTA)

Augmentations shouldn't just be for training. At inference, if the model predicts on exactly one central crop of an image, it is vulnerable to positional variation. 

**Test-Time Augmentation (TTA):** Generate $K$ conditionally augmented versions of the input image (e.g., center crop, 4 corners, horizontal flips), predict on all $K$ views, and mathematically average the Softmax probability vectors.

This provides a completely **free accuracy improvement** without any retraining. 
Benchmarking on ImageNet typically shows:
*   **5-crop TTA:** $\sim +0.5\%$ accuracy.
*   **10-crop TTA:** $\sim +1.2\%$ accuracy for ResNet-50.

**The Tradeoff:**
10-crop TTA requires 10 forward passes through the network. The inference latency increases by nearly $10\times$. You must balance the Accuracy/Latency Pareto curve strictly based on hardware constraints.


In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: TTA Inference Wrapper
# -----------------------------------------------------
import torchvision.transforms.functional as TF

def tta_predict_10_crop(model, image, crop_size=224):
    '''
    Applies standard 10-crop TTA (4 corners + 1 center) x (original + horizontal flip)
    '''
    model.eval()
    B, C, H, W = image.shape
    
    # Generate spatial boundaries for corner crops
    crops = [
        (0, 0),                                # Top left
        (0, W - crop_size),                    # Top right
        (H - crop_size, 0),                    # Bottom left
        (H - crop_size, W - crop_size),        # Bottom right
        ((H - crop_size)//2, (W - crop_size)//2) # Center
    ]
    
    predictions = []
    with torch.no_grad():
        for is_flipped in [False, True]:
            img_process = TF.hflip(image) if is_flipped else image
            for y, x in crops:
                crop = img_process[:, :, y:y+crop_size, x:x+crop_size]
                pred = torch.softmax(model(crop), dim=1)
                predictions.append(pred)
                
    # Average the 10 prediction vectors
    avg_prediction = torch.stack(predictions).mean(dim=0)
    return avg_prediction

# TEST IT
class DummyModel(torch.nn.Module):
    def forward(self, x): return torch.randn(x.size(0), 10) # 10 classes

dummy_img = torch.randn(1, 3, 256, 256)
model = DummyModel()
tta_out = tta_predict_10_crop(model, dummy_img, 224)

print(f"TTA Averaged Output Shape: {tta_out.shape} - Ready for argmax")


### COMMON PITFALLS
*   **Incompatible TTA Operators:** Applying Hue shifting or Scale jittering at TTA can completely break detection accuracy since you change the physical structure or color of the objects out of distribution. Typical TTA is strictly limited to deterministic spatial crops/flips or multi-scale pyramids.


## 6. Class Imbalance Handling

Real factory datasets have 10,000 "good product" images and 50 "defective" images. Standard training completely ignores the defectives. Here are three mathematical strategies to enforce minority visibility.

### 1. WeightedRandomSampler
Assign each sample a weight inversely proportional to class frequency. PyTorch's DataLoader will mathematically balance the sampling odds per epoch.

### 2. Focal Loss (Re-implemented)
Down-weights correctly classified background samples using an exponential decay $(1 - p_t)^\gamma$. For a 99% confident background prediction where $\gamma=2$, the loss weight becomes $(1 - 0.99)^2 = 0.0001$, clearing the gradient runway for the defective samples.

### 3. Class-Weighted Cross Entropy 
Using `nn.CrossEntropyLoss(weight=class_weights)`. Mathematically identical to oversampling minority classes, it multiplies the raw gradient of minority samples by $1.0 / \text{freq}$.


In [ ]:
from torch.utils.data import WeightedRandomSampler

# -----------------------------------------------------
# IMPLEMENTATION: Strategies
# -----------------------------------------------------

# 1. Sampler
class_counts = [1000, 10]
class_weights = [1.0 / count for count in class_counts]
sample_labels = [0]*1000 + [1]*10  # 1010 elements
sample_weights = [class_weights[label] for label in sample_labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# 2. Focal Loss Context Module
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        # inputs are logits
        ce_loss = torch.nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss) 
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# 3. Class Weights Equivalent
tensor_class_weights = torch.tensor(class_weights, dtype=torch.float)
ce_weighted = torch.nn.CrossEntropyLoss(weight=tensor_class_weights)

# TEST IT 
logits = torch.randn(4, 2)
targets = torch.tensor([0, 0, 1, 0])
fl = FocalLoss()
print(f"Focal Loss Output: {fl(logits, targets).item():.4f}")
print(f"Weighted CE Loss: {ce_weighted(logits, targets).item():.4f}")

# Evaluation Note: Without handling, the confusion matrix for Class 1 (Minority) will be [0, 10] (100% False Negative Rate). With Focal Loss or Sampling, it shifts heavily toward correct diagonals.


### COMMON PITFALLS
*   **Balancing Validation Sets:** You ONLY balance the training dataloader. Balancing or resampling the validation set mathematically destroys your metrics since it no longer represents the real-world operational domain distributions.


## 7. MixUp & CutMix Regularization

**MixUp:** You take an image of a Dog, and an image of a Car. You mathematically blend them together (e.g., $60\%$ opacity Dog, $40\%$ opacity Car). The target label becomes exactly `[0.6 Dog, 0.4 Car]`. This forces the network to stop relying on binary "all or nothing" spatial features and learn continuous probability boundaries.

**CutMix:** You physically cut a square box out of the Car image and paste it directly on top of the Dog's face. The label is proportional to the area. 

**Why CutMix is genius:** If a CNN is trying to detect a Dog, it usually just learns to look for the Dog's eyes and ears. By pasting a car over the dog's face, the CNN is forced to look at the dog's paws, tail, and fur to make the prediction. It forces the network to learn holistic, full-body feature representations, acting as the ultimate spatial regularization.


In [ ]:
import numpy as np

# IMPLEMENTATION: CutMix Mathematical Core Calculation

def generate_cutmix_box(img_shape, lambda_val):
    B, C, H, W = img_shape
    cut_rat = np.sqrt(1. - lambda_val)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    # uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

# TEST IT
B, C, H, W = 4, 3, 224, 224
lam = np.random.beta(1.0, 1.0) # Beta distribution scaling factor
bbx1, bby1, bbx2, bby2 = generate_cutmix_box((B, C, H, W), lam)
print(f"Generated CutMix paste coordinates: ({bbx1}, {bby1}) to ({bbx2}, {bby2}) for lambda {lam:.2f}")


### COMMON PITFALLS
*   **Cutting out the primary object entirely:** Since CutMix cuts randomly, you might entirely cut out a tiny insect and replace it with background from another class, but mathematically label it 60% insect still. This introduces false-label noise. The network tolerates it, but bounding box aware-cutting works better.
